# Homework 02: Vector Search

## Modules import 

In [1]:
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

## Q1. Embedding a query

In [2]:
embedder = Embedder(path="models/Xenova/all-MiniLM-L6-v2")
q1_vector = embedder.encode("How does approximate nearest neighbor search work?")
print(f"Vector len: {len(q1_vector)}")
print(f"Value q1_vector[0]: {q1_vector[0]:.4f}")

Vector len: 384
Value q1_vector[0]: -0.0206


## Loading the data

Using commit `8c1834d`

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Docs loaded: {len(documents)}")
print(f"Sample: {documents[0]}")

Docs loaded: 72
Sample: {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.

## Q2. Cosine similarity

In [4]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc["filename"] == target_file)
print(f"Found doc: {target_doc['filename']}")

# print(f'Content: {target_doc["content"]}')

doc_vector = embedder.encode(target_doc["content"])
similarity = np.dot(q1_vector, doc_vector)
print(f"Cosine similarity: {similarity:.4f}")

Found doc: 02-vector-search/lessons/07-sqlitesearch-vector.md
Cosine similarity: 0.3611


## Q3. Chunking and search by hand

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Chunks: {len(chunks)}")

# Кодируем содержимое всех чанков
chunk_contents = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunk_contents)
print(f"X shape: {X.shape}")

# Вычисляем скоры (скалярное произведение)
scores = X.dot(q1_vector)
best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]
print(f"Best chunk index: {best_idx}")
print(f"Best chunk score: {scores[best_idx]:.4f}")
print(f"Best chunk file: {best_chunk['filename']}")

Chunks: 295
X shape: (295, 384)
Best chunk index: 94
Best chunk score: 0.6489
Best chunk file: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

In [6]:
v_index = VectorSearch(keyword_fields=["filename"])
v_index.append_batch(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

# Поиск
q4_results = v_index.search(q4_vector, num_results=5)
print("VectorSearch top-5:")
for i, res in enumerate(q4_results):
    print(f"{i + 1}. {res['filename']} (start: {res.get('start')})")

VectorSearch top-5:
1. 04-evaluation/lessons/05-search-metrics.md (start: 0)
2. 04-evaluation/lessons/01-intro.md (start: 0)
3. 01-agentic-rag/lessons/05-search.md (start: 0)
4. 04-evaluation/lessons/01-intro.md (start: 2000)
5. 04-evaluation/lessons/15-next-steps.md (start: 0)


## Q5. Text search vs vector search

In [7]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)

vector_results = v_index.search(q5_vector, num_results=5)
text_results = text_index.search(query=q5_query, num_results=5)

vector_filenames = [res["filename"] for res in vector_results]
text_filenames = [res["filename"] for res in text_results]

print("VectorSearch top-5:")
for i, fn in enumerate(vector_filenames):
    print(f" {i + 1}. {fn}")

print("\nText search top-5:")
for i, fn in enumerate(text_filenames):
    print(f" {i + 1}. {fn}")

diff = set(vector_filenames) - set(text_filenames)
print(f"\nFiles in Vector Search only: {diff}")

VectorSearch top-5:
 1. 02-vector-search/lessons/08-pgvector.md
 2. 02-vector-search/lessons/08-pgvector.md
 3. 03-orchestration/lessons/05-rag.md
 4. 02-vector-search/lessons/08-pgvector.md
 5. 02-vector-search/lessons/08-pgvector.md

Text search top-5:
 1. 02-vector-search/lessons/02-embeddings.md
 2. 03-orchestration/lessons/05-rag.md
 3. 02-vector-search/lessons/01-intro.md
 4. 03-orchestration/lessons/05-rag.md
 5. 02-vector-search/lessons/01-intro.md

Files in Vector Search only: {'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid search


In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)

vector_results = v_index.search(q6_vector, num_results=10)
text_results = text_index.search(q6_query, num_results=10)

results = rrf([vector_results, text_results])

print("Hybrid search RRF top-5:")
for rank, res in enumerate(results):
    print(f"{rank + 1}. {res['filename']} (start: {res.get('start', 0)})")

Hybrid search RRF top-5:
1. 01-agentic-rag/lessons/13-function-calling.md (start: 4000)
2. 01-agentic-rag/lessons/13-function-calling.md (start: 2000)
3. 01-agentic-rag/lessons/01-intro.md (start: 2000)
4. 01-agentic-rag/lessons/14-agentic-loop.md (start: 0)
5. 04-evaluation/lessons/02-ground-truth.md (start: 1000)
